# SDC — Trích xuất feature DHCP / DNS / mDNS / TLS từ CIC IoT Dataset 2022

Bước đầu tiên là viết extractor
đọc pcap và xuất ra bảng feature thô (1 row / event) cho từng nguồn:

| Nguồn | Field | Ghi chú |
|---|---|---|
| DHCP | hostname, vendor_class_id, param_req_list (option order), message type | Discover/Offer/Request/ACK/NAK |
| DNS | qry.name, qry.type, rcode | UDP/53, không tính mDNS |
| mDNS | qry.name, qry.type | UDP/5353 |
| TLS | SNI, ALPN, cipher suites, version | Từ ClientHello, có TCP reassembly và loại GREASE |

**Không dùng CICFlowMeter/tshark** (không có sẵn trong môi trường, không có quyền cài qua apt) — dùng `scapy`
(pip, không cần binary hệ thống) để đọc pcap và tự parse thủ công layer TLS record (scapy chưa hỗ trợ tốt
dissect ClientHello nhẹ mà không cần load thêm crypto).

Output: 3 bảng CSV thô (`dhcp_features.csv`, `dns_features.csv`, `tls_features.csv`) trong `Data/features/`,
mỗi row gắn kèm metadata thiết bị (category, device, scenario, capture file) để bước sau (coverage check,
labeling, train) dùng lại.

In [ ]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

In [ ]:
# Nạp định nghĩa từ 03 mà không chạy train.
from pathlib import Path

_cwd = Path.cwd().resolve()
_ROOT = next((p for p in (_cwd, *_cwd.parents)
              if (p / "Code" / "03_train_model.ipynb").is_file()), None)
assert _ROOT is not None, f"Không thấy Code/03_train_model.ipynb quanh {_cwd}"
_CODE = _ROOT / "Code"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_CODE / "03_train_model.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import pandas as pd
from IPython.display import display

In [ ]:
import os
import struct
import glob
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from scapy.all import PcapReader, DHCP, BOOTP, DNS, DNSQR, UDP, TCP, IP, Raw

BASE_DIR = ROOT / "Data" / "1-Power"
OUT_DIR = ROOT / "Data" / "features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert BASE_DIR.exists(), f"Không tìm thấy {BASE_DIR.resolve()} — giải nén 1-Power.tar.gz trước"

## 1. Liệt kê capture files

Cấu trúc thư mục: `1-Power/<Category>/<Device>/POWER/<file>.pcap`. Bỏ qua file rác AppleDouble (`._*.pcap`,
sinh ra khi giải nén archive tạo trên macOS — không phải pcap thật).

In [ ]:
def iter_captures(base_dir: Path):
    """Yield (category, device, scenario, pcap_path) for every real capture file."""
    for pcap_path in sorted(base_dir.glob("*/*/*/*.pcap")):
        if pcap_path.name.startswith("._"):
            continue
        scenario_dir = pcap_path.parent          # e.g. POWER
        device_dir = scenario_dir.parent          # e.g. Amazon Echo Dot 1
        category_dir = device_dir.parent          # e.g. Audio
        yield category_dir.name, device_dir.name, scenario_dir.name, pcap_path

captures = list(iter_captures(BASE_DIR))
print(f"{len(captures)} capture file(s) hop le")
pd.DataFrame(captures, columns=["category", "device", "scenario", "path"]).head(10)

## 2. DHCP extractor

Field cần: `bootp.option.hostname`, `bootp.option.vendor_class_id`, `bootp.option.list` (thứ tự option —
Parameter Request List, option 55) + message type (Discover/Offer/Request/ACK/NAK).

## 3. DNS / mDNS extractor

Phân biệt DNS (UDP/53) và mDNS (UDP/5353) qua port. Xuất từng query record (`dns.qry.name`, `dns.qry.type`)
và `rcode` khi packet là response.

## 4. TLS ClientHello extractor

Dùng parser chung trong `Code/03_train_model.ipynb`, có TCP reassembly và loại GREASE trước khi tạo fingerprint.

## 5. Chạy extractor trên toàn bộ capture

Đọc từng pcap theo kiểu streaming (`PcapReader`) để tránh load nguyên file lớn (~270MB) vào RAM cùng lúc.
Mỗi row được gắn thêm metadata thiết bị (`category`, `device`, `scenario`, `capture_file`).

In [ ]:
def meta(category, device, scenario, pcap_path):
    return {
        "category": category,
        "device": device,
        "scenario": scenario,
        "capture_file": pcap_path.name,
        "session_id": f"{device}_{scenario}_{pcap_path.stem}",
    }

dhcp_rows, dns_rows, tls_rows = [], [], []
capture_summary = []

for category, device, scenario, pcap_path in tqdm(captures, desc="capture files"):
    m = meta(category, device, scenario, pcap_path)
    n_dhcp = n_dns = n_mdns = n_tls = 0
    device_mac = infer_device_mac(pcap_path)
    if device_mac is None:
        raise RuntimeError(f"Không suy được MAC thiết bị từ {pcap_path}")
    tls_stream = TLSStreamReassembler()

    with PcapReader(str(pcap_path)) as reader:
        for pkt in reader:
            ts = float(pkt.time)

            dhcp_row = extract_dhcp(pkt, ts)
            if dhcp_row and dhcp_matches_device(pkt, device_mac):
                dhcp_rows.append({**m, **dhcp_row})
                n_dhcp += 1

            if not packet_from_device(pkt, device_mac):
                continue

            for dns_row in extract_dns(pkt, ts):
                dns_rows.append({**m, **dns_row})
                if dns_row["is_mdns"]:
                    n_mdns += 1
                else:
                    n_dns += 1

            tls_row = extract_tls(pkt, ts, tls_stream)
            if tls_row:
                tls_rows.append({**m, **tls_row})
                n_tls += 1

    capture_summary.append({**m, "n_dhcp": n_dhcp, "n_dns": n_dns, "n_mdns": n_mdns, "n_tls": n_tls})

dhcp_df = pd.DataFrame(dhcp_rows)
dns_df = pd.DataFrame(dns_rows)
tls_df = pd.DataFrame(tls_rows)
summary_df = pd.DataFrame(capture_summary)

print("dhcp_df:", dhcp_df.shape, "dns_df:", dns_df.shape, "tls_df:", tls_df.shape)

## 6. Lưu output

Ghi 3 bảng feature thô + bảng summary (số event/nguồn mỗi capture — dùng làm input thô cho bước coverage
check ở mục 4 của kế hoạch) ra `Data/features/`.

In [ ]:
dhcp_df.to_csv(OUT_DIR / "dhcp_features.csv", index=False)
dns_df.to_csv(OUT_DIR / "dns_features.csv", index=False)
tls_df.to_csv(OUT_DIR / "tls_features.csv", index=False)
summary_df.to_csv(OUT_DIR / "capture_summary.csv", index=False)

print("Da luu vao", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*.csv")):
    print(" -", f.name)

## 7. Kiểm tra nhanh (preview)

Xem nhanh số lượng event theo nguồn / category / device để có cảm nhận ban đầu về độ phủ trước khi làm
coverage check đầy đủ (mục 4 của kế hoạch).

In [ ]:
print("Capture co it nhat 1 nguon feature:")
has_any = summary_df.assign(
    has_dhcp=summary_df.n_dhcp > 0,
    has_dns=summary_df.n_dns > 0,
    has_mdns=summary_df.n_mdns > 0,
    has_tls=summary_df.n_tls > 0,
)
has_any["n_sources"] = has_any[["has_dhcp", "has_dns", "has_mdns", "has_tls"]].sum(axis=1)
display(has_any.groupby("category")[["has_dhcp", "has_dns", "has_mdns", "has_tls"]].mean().round(2))

In [ ]:
print("Phan bo so nguon feature co mat / capture:")
display(has_any["n_sources"].value_counts().sort_index())

## 8. Mở rộng: xử lý Idle dataset (toàn mạng, gắn nhãn qua MAC)

Khác với Power (mỗi thiết bị 1 file riêng), Idle là **1 file/đêm × 30 đêm**, chứa traffic của cả 40 thiết bị
trộn chung trên cùng mạng (xem `Data/2-Idle/Readme.txt`). Không còn metadata tên thiết bị theo path — gắn
nhãn bằng cách join **Ethernet source MAC** của từng packet với bảng ground-truth `Search/Device List.xlsx`
(40 thiết bị, đã xác nhận khớp 100% MAC khi kiểm tra sample).

Vì capture liên tục cả đêm (không có ranh giới "1 lần bật nguồn = 1 session" như Power), mỗi event được gắn
thêm `session_id = <device>_<ngày>_<giờ UTC>` (bucket theo giờ) để làm đơn vị session cho bước tổng hợp
feature/coverage check sau này — cùng tinh thần "1 row/session" của kế hoạch, áp dụng cho dữ liệu liên tục.

In [ ]:
import re
from datetime import datetime, timezone

IDLE_DIR = ROOT / "Data" / "2-Idle"

device_list_df = pd.read_excel(ROOT / "Search" / "Device List.xlsx")
device_list_df["Category"] = device_list_df["Category"].ffill()

mac_to_device = {}
mac_to_category = {}
for _, row in device_list_df.iterrows():
    mac = str(row["MAC Address"]).strip().lower()
    mac_to_device[mac] = row["Device Name"]
    mac_to_category[mac] = row["Category"]

known_macs = set(mac_to_device)
print(f"{len(known_macs)} thiet bi trong bang MAC ground-truth")

In [ ]:
def idle_date_from_filename(pcap_path: Path) -> str:
    m = re.match(r"(\d{4})_(\d{2})_(\d{2})_Idle", pcap_path.stem)
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

idle_files = sorted(IDLE_DIR.glob("*_Idle.pcap"))
print(f"{len(idle_files)} idle capture file(s)")
[(f.name, idle_date_from_filename(f)) for f in idle_files[:3]]

### Vòng lặp trích xuất Idle

Tái sử dụng nguyên `extract_dhcp` / `extract_dns` / `extract_tls` đã viết ở mục 2-4. Chỉ khác: lọc theo
`Ether.src` nằm trong `known_macs` trước khi trích xuất (vừa để gắn nhãn thiết bị đúng, vừa bỏ qua sớm phần
traffic không phải từ 1 trong 40 thiết bị — ví dụ router/AP — để đỡ tốn thời gian xử lý).

In [ ]:
from scapy.all import Ether

idle_dhcp_rows, idle_dns_rows, idle_tls_rows = [], [], []
idle_capture_summary = []

for pcap_path in tqdm(idle_files, desc="idle capture files"):
    date_str = idle_date_from_filename(pcap_path)
    per_device_counts = {}  # mac -> {n_dhcp, n_dns, n_mdns, n_tls}
    tls_streams = {}

    with PcapReader(str(pcap_path)) as reader:
        for pkt in reader:
            if not pkt.haslayer(Ether):
                continue
            ts = float(pkt.time)
            src_mac = packet_source_mac(pkt)
            dhcp_row = extract_dhcp(pkt, ts)
            dhcp_mac = dhcp_row.get("client_mac") if dhcp_row else None
            owner_mac = src_mac if src_mac in known_macs else dhcp_mac
            if owner_mac not in known_macs:
                continue

            hour = datetime.fromtimestamp(ts, tz=timezone.utc).hour
            device = mac_to_device[owner_mac]
            category = mac_to_category[owner_mac]
            m = {
                "category": category,
                "device": device,
                "scenario": "IDLE",
                "capture_file": pcap_path.name,
                "date": date_str,
                "session_id": f"{device}_{date_str}_{hour:02d}",
            }

            counts = per_device_counts.setdefault(
                owner_mac, {"n_dhcp": 0, "n_dns": 0, "n_mdns": 0, "n_tls": 0}
            )

            if dhcp_row and dhcp_matches_device(pkt, owner_mac):
                idle_dhcp_rows.append({**m, **dhcp_row})
                counts["n_dhcp"] += 1

            if src_mac != owner_mac:
                continue

            for dns_row in extract_dns(pkt, ts):
                idle_dns_rows.append({**m, **dns_row})
                if dns_row["is_mdns"]:
                    counts["n_mdns"] += 1
                else:
                    counts["n_dns"] += 1

            tls_stream = tls_streams.setdefault(owner_mac, TLSStreamReassembler())
            tls_row = extract_tls(pkt, ts, tls_stream)
            if tls_row:
                idle_tls_rows.append({**m, **tls_row})
                counts["n_tls"] += 1

    for mac, counts in per_device_counts.items():
        idle_capture_summary.append({
            "category": mac_to_category[mac],
            "device": mac_to_device[mac],
            "scenario": "IDLE",
            "capture_file": pcap_path.name,
            "date": date_str,
            **counts,
        })

idle_dhcp_df = pd.DataFrame(idle_dhcp_rows)
idle_dns_df = pd.DataFrame(idle_dns_rows)
idle_tls_df = pd.DataFrame(idle_tls_rows)
idle_summary_df = pd.DataFrame(idle_capture_summary)

print("idle_dhcp_df:", idle_dhcp_df.shape, "idle_dns_df:", idle_dns_df.shape, "idle_tls_df:", idle_tls_df.shape)

## 9. Lưu output Idle + gộp chung với Power

In [ ]:
idle_dhcp_df.to_csv(OUT_DIR / "dhcp_features_idle.csv", index=False)
idle_dns_df.to_csv(OUT_DIR / "dns_features_idle.csv", index=False)
idle_tls_df.to_csv(OUT_DIR / "tls_features_idle.csv", index=False)
idle_summary_df.to_csv(OUT_DIR / "capture_summary_idle.csv", index=False)

print("Da luu Idle features vao", OUT_DIR.resolve())
for f in ["dhcp_features_idle.csv", "dns_features_idle.csv", "tls_features_idle.csv", "capture_summary_idle.csv"]:
    print(" -", f)

In [ ]:
combined_dhcp_df = pd.concat([dhcp_df, idle_dhcp_df], ignore_index=True)
combined_dns_df = pd.concat([dns_df, idle_dns_df], ignore_index=True)
combined_tls_df = pd.concat([tls_df, idle_tls_df], ignore_index=True)
combined_summary_df = pd.concat([summary_df, idle_summary_df], ignore_index=True)

combined_dhcp_df.to_csv(OUT_DIR / "dhcp_features_all.csv", index=False)
combined_dns_df.to_csv(OUT_DIR / "dns_features_all.csv", index=False)
combined_tls_df.to_csv(OUT_DIR / "tls_features_all.csv", index=False)
combined_summary_df.to_csv(OUT_DIR / "capture_summary_all.csv", index=False)

print("combined_dhcp_df:", combined_dhcp_df.shape)
print("combined_dns_df:", combined_dns_df.shape)
print("combined_tls_df:", combined_tls_df.shape)
print("Da luu ban gop (Power+Idle) vao", OUT_DIR.resolve())

## 10. Kiểm tra nhanh coverage Idle (so với Power ở mục 7)

In [ ]:
idle_has_any = idle_summary_df.assign(
    has_dhcp=idle_summary_df.n_dhcp > 0,
    has_dns=idle_summary_df.n_dns > 0,
    has_mdns=idle_summary_df.n_mdns > 0,
    has_tls=idle_summary_df.n_tls > 0,
)
print("Idle - ty le (thiet-bi x ngay) co tung nguon, theo category:")
display(idle_has_any.groupby("category")[["has_dhcp", "has_dns", "has_mdns", "has_tls"]].mean().round(2))

In [ ]:
print("Idle - so ngay quan sat duoc it nhat 1 event / thiet bi (top 10 nhieu nhat):")
display(idle_summary_df.groupby("device").size().sort_values(ascending=False).head(10))

print()
print("Idle - so ngay quan sat duoc it nhat 1 event / thiet bi (bottom 10 it nhat, co the la thiet bi im lang):")
display(idle_summary_df.groupby("device").size().sort_values().head(10))